# SAE-XCrash — NB02: Modeling Baselines v2
**Work Package WP2 — Addendum** | P11: LightGBM Ablation | P10: Geographic Hold-Out

Load-and-continue notebook. Run **after** NB01_Data_Protocol_v2.
All outputs go to `logs/wp2_v2/` and `figures/wp2_v2/`.


---
## Step 1 — Environment & Paths

In [ ]:
!pip install -q xgboost lightgbm scikit-learn pyarrow matplotlib numpy

import os, json, time, pickle
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJ_ROOT  = Path('/content/drive/MyDrive/SAE-XCrash')

# ── Exact paths confirmed from NB01_v2 run ────────────────────────────────────
XGB_MODEL_PATH  = PROJ_ROOT / 'models' / 'wp2' / 'xgb_best.json'
LGBM_MODEL_PATH = PROJ_ROOT / 'models' / 'wp2' / 'lgbm_best.txt'
WP2_META_PATH   = PROJ_ROOT / 'logs' / 'wp2_meta.json'
HP_LOG_PATH     = PROJ_ROOT / 'logs' / 'wp2_hyperparameter_log.json'
PROC_DIR        = PROJ_ROOT / 'data' / 'processed'
TRAIN_PARQUET   = PROC_DIR / 'usa_train_processed.parquet'
VAL_PARQUET     = PROC_DIR / 'usa_val_processed.parquet'
TEST_PARQUET    = PROC_DIR / 'usa_test_processed.parquet'

# ── New versioned output dirs ─────────────────────────────────────────────────
LOGS_DIR = PROJ_ROOT / 'logs'    / 'wp2_v2'
FIGS_DIR = PROJ_ROOT / 'figures' / 'wp2_v2'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DPI  = 300
print(f"XGB model  : {XGB_MODEL_PATH}")
print(f"LGBM model : {LGBM_MODEL_PATH}")
print(f"Test data  : {TEST_PARQUET}")
print(f"Outputs    : {LOGS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
XGB model  : /content/drive/MyDrive/SAE-XCrash/models/wp2/xgb_best.json
LGBM model : /content/drive/MyDrive/SAE-XCrash/models/wp2/lgbm_best.txt
Test data  : /content/drive/MyDrive/SAE-XCrash/data/processed/usa_test_processed.parquet
Outputs    : /content/drive/MyDrive/SAE-XCrash/logs/wp2_v2


---
## Step 2 — Load Artifacts

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
print("Loading XGBoost model ...")
xgb_booster = xgb.Booster()
xgb_booster.load_model(str(XGB_MODEL_PATH))
print(f"  Loaded ✓  ({XGB_MODEL_PATH.stat().st_size/1e6:.1f} MB)")

# ── LightGBM ──────────────────────────────────────────────────────────────────
print("Loading LightGBM model ...")
if LGBM_MODEL_PATH.suffix == '.txt':
    lgbm_booster = lgb.Booster(model_file=str(LGBM_MODEL_PATH))
else:
    with open(LGBM_MODEL_PATH, 'rb') as f:
        lgbm_booster = pickle.load(f)
print(f"  Loaded ✓  ({LGBM_MODEL_PATH.stat().st_size/1e6:.1f} MB)")

# ── WP2 metadata ──────────────────────────────────────────────────────────────
print("Loading WP2 metadata ...")
with open(WP2_META_PATH) as f:
    wp2_meta = json.load(f)
# FEAT_USA from wp2_meta has 32 features (pre-NB01_v2).
# The updated parquets now also have s_street_te and s_city_te (34 features).
# We rebuild the feature list from the test parquet to pick up the new columns.
FEAT_USA_V1 = wp2_meta['feat_usa']   # original 32 features
SPW          = wp2_meta.get('spw', 36.0)
print(f"  V1 features: {len(FEAT_USA_V1)},  SPW: {SPW:.1f}")

# ── Test split (updated by NB01_v2 — now has s_street_te, s_city_te) ──────────
print("Loading test split ...")
usa_test = pd.read_parquet(TEST_PARQUET)
EXCL     = {'label','ID','Start_Time','datetime',
            'accident_index','accident_reference','collision_reference','row_id'}
FEAT_USA = [c for c in usa_test.columns
            if c.startswith(('t_','w_','r_','s_')) and c not in EXCL]
print(f"  Test: {len(usa_test):,} rows,  {len(FEAT_USA)} features")
print(f"  New features vs V1: {set(FEAT_USA) - set(FEAT_USA_V1)}")

# Models were trained on 32 features (FEAT_USA_V1).
# The parquet now has 34 after NB01_v2 added s_street_te and s_city_te.
# Use FEAT_USA_V1 for all predictions — new features only matter if models are retrained.
X_test     = usa_test[FEAT_USA_V1].values.astype(np.float32)
FEAT_USA   = FEAT_USA_V1   # alias so downstream code is unchanged
y_test     = usa_test['label'].values
feat_means = X_test.mean(axis=0).astype(np.float32)
no_skill   = float(y_test.mean())
print(f"  Using {len(FEAT_USA)} features (original model features)")
print(f"  No-skill AUPRC: {no_skill:.4f}")

# ── Feature groups ────────────────────────────────────────────────────────────
feat_groups = {
    'Temporal': [f for f in FEAT_USA_V1 if f.startswith('t_')],
    'Weather':  [f for f in FEAT_USA_V1 if f.startswith('w_')],
    'Road':     [f for f in FEAT_USA_V1 if f.startswith('r_')],
    'Spatial':  [f for f in FEAT_USA_V1 if f.startswith('s_')],
}
print(f"  Groups: { {k: len(v) for k, v in feat_groups.items()} }")

# ── Predict helpers ───────────────────────────────────────────────────────────
def xgb_predict(X):
    return xgb_booster.predict(xgb.DMatrix(X))

def lgbm_predict(X):
    return (lgbm_booster.predict(X) if isinstance(lgbm_booster, lgb.Booster)
            else lgbm_booster.predict_proba(X)[:, 1])

# ── Hyperparameter log (for geo hold-out) ─────────────────────────────────────
if HP_LOG_PATH.exists():
    with open(HP_LOG_PATH) as f:
        hp_log = json.load(f)
    xgb_best_params = hp_log.get('xgb_best', {})
    print(f"  XGB best params loaded: {list(xgb_best_params.keys())}")
else:
    print(f"  HP log not found at {HP_LOG_PATH} — using fallback params")
    xgb_best_params = {
        'learning_rate': 0.063, 'n_estimators': 1100, 'max_depth': 10,
        'min_child_weight': 9, 'subsample': 0.945, 'colsample_bytree': 0.728,
        'reg_alpha': 5.6e-6, 'reg_lambda': 1.3e-7,
    }

print("\nAll artifacts loaded ✓")


Loading XGBoost model ...
  Loaded ✓  (38.2 MB)
Loading LightGBM model ...
  Loaded ✓  (49.7 MB)
Loading WP2 metadata ...
  V1 features: 32,  SPW: 36.0
Loading test split ...
  Test: 166,552 rows,  34 features
  New features vs V1: {'s_city_te', 's_street_te'}
  Using 32 features (original model features)
  No-skill AUPRC: 0.0291
  Groups: {'Temporal': 8, 'Weather': 10, 'Road': 12, 'Spatial': 2}
  XGB best params loaded: ['max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'min_child_weight', 'reg_alpha', 'reg_lambda', 'gamma', 'n_estimators']

All artifacts loaded ✓


---
## Step 3 — P11: LightGBM Feature Group Ablation
Mirrors Table 5 (XGBoost ablation). Forward passes only — no retraining.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# P11 — LightGBM Feature Group Ablation
# ─────────────────────────────────────────────────────────────────────────────
print("P11 — LightGBM Feature Group Ablation")
print("=" * 70)

ablation_configs = [
    ('Full model (all features)',   None,     None),
    ('Remove Temporal (T)',         'remove', feat_groups['Temporal']),
    ('Remove Weather (W)',          'remove', feat_groups['Weather']),
    ('Remove Road/Context (R)',     'remove', feat_groups['Road']),
    ('Remove Spatial (S)',          'remove', feat_groups['Spatial']),
    ('Temporal only',               'keep',   feat_groups['Temporal']),
    ('Weather + Road only',         'keep',   feat_groups['Weather'] + feat_groups['Road']),
]

full_auprc_xgb  = average_precision_score(y_test, xgb_predict(X_test))
full_auprc_lgbm = average_precision_score(y_test, lgbm_predict(X_test))

print(f"  No-skill AUPRC : {no_skill:.4f}")
print(f"  XGBoost full   : {full_auprc_xgb:.4f}  (paper Table 5 reference: 0.1306)")
print(f"  LightGBM full  : {full_auprc_lgbm:.4f}  (paper Table 5 reference: 0.1311)")

xgb_abl, lgbm_abl = [], []

for label, mode, feat_list in ablation_configs:
    X_abl = X_test.copy()
    if mode == 'remove':
        idx = [FEAT_USA.index(f) for f in feat_list if f in FEAT_USA]
        X_abl[:, idx] = feat_means[idx]
    elif mode == 'keep':
        keep = set(FEAT_USA.index(f) for f in feat_list if f in FEAT_USA)
        mask = [i for i in range(len(FEAT_USA)) if i not in keep]
        X_abl[:, mask] = feat_means[mask]

    ax = average_precision_score(y_test, xgb_predict(X_abl))
    al = average_precision_score(y_test, lgbm_predict(X_abl))
    xgb_abl.append( {'ablation': label, 'auprc': round(ax, 4), 'delta': round(ax - full_auprc_xgb, 4)})
    lgbm_abl.append({'ablation': label, 'auprc': round(al, 4), 'delta': round(al - full_auprc_lgbm, 4)})

# ── Print comparison table ─────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"{'Ablation':<40} {'XGB AUPRC':>10} {'XGB Δ':>8} {'LGB AUPRC':>10} {'LGB Δ':>8}")
print("-" * 80)
for xr, lr in zip(xgb_abl, lgbm_abl):
    print(f"  {xr['ablation']:<38} {xr['auprc']:>10.4f} {xr['delta']:>+8.4f} "
          f"{lr['auprc']:>10.4f} {lr['delta']:>+8.4f}")

# ── Save ───────────────────────────────────────────────────────────────────────
with open(LOGS_DIR / 'ablation_xgb_results.json', 'w') as f:
    json.dump(xgb_abl, f, indent=2)
with open(LOGS_DIR / 'ablation_lgbm_results.json', 'w') as f:
    json.dump(lgbm_abl, f, indent=2)
print(f"\nSaved to {LOGS_DIR} ✓")
print("P11 complete ✓")


P11 — LightGBM Feature Group Ablation
  No-skill AUPRC : 0.0291
  XGBoost full   : 0.1341  (paper Table 5 reference: 0.1306)
  LightGBM full  : 0.1340  (paper Table 5 reference: 0.1311)

Ablation                                  XGB AUPRC    XGB Δ  LGB AUPRC    LGB Δ
--------------------------------------------------------------------------------
  Full model (all features)                  0.1341  +0.0000     0.1340  +0.0000
  Remove Temporal (T)                        0.1246  -0.0095     0.1201  -0.0139
  Remove Weather (W)                         0.1255  -0.0086     0.1265  -0.0075
  Remove Road/Context (R)                    0.1352  +0.0012     0.0722  -0.0618
  Remove Spatial (S)                         0.0315  -0.1025     0.0299  -0.1042
  Temporal only                              0.0366  -0.0974     0.0310  -0.1030
  Weather + Road only                        0.0300  -0.1041     0.0289  -0.1051

Saved to /content/drive/MyDrive/SAE-XCrash/logs/wp2_v2 ✓
P11 complete ✓


---
## Step 4 — P10: Geographic Hold-Out
Train XGBoost on all non-CA/TX/FL states (2016–2021). Test on CA+TX+FL in 2023 only.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# P10 — Geographic Hold-Out
# ─────────────────────────────────────────────────────────────────────────────
print("P10 — Geographic Hold-Out Experiment")
print("=" * 60)

HOLDOUT_STATES = ['CA', 'TX', 'FL']

# ── Load all splits (train parquet has temporal info) ─────────────────────────
print("Loading train + val + test parquets ...")
usa_train = pd.read_parquet(TRAIN_PARQUET)
usa_val   = pd.read_parquet(VAL_PARQUET)
df_all    = pd.concat([usa_train, usa_val, usa_test], ignore_index=True)
print(f"  Combined: {len(df_all):,} rows")
del usa_train, usa_val   # free memory

# ── Reconstruct state string from s_state code ────────────────────────────────
# NB01 label-encoded states alphabetically; same 49-state list
US_STATES = sorted([
    'AL','AR','AZ','CA','CO','CT','DC','DE','FL','GA','HI','IA','ID',
    'IL','IN','KS','KY','LA','MA','MD','ME','MI','MN','MO','MS','MT',
    'NC','ND','NE','NH','NJ','NM','NV','NY','OH','OK','OR','PA','RI',
    'SC','SD','TN','TX','UT','VA','VT','WA','WI','WV','WY'
])
code_to_state = {i: s for i, s in enumerate(US_STATES)}
state_to_code = {s: i for i, s in code_to_state.items()}
holdout_codes = set(state_to_code[s] for s in HOLDOUT_STATES if s in state_to_code)
print(f"  Holdout codes: {HOLDOUT_STATES} → {holdout_codes}")

# Verify against data
found_codes = holdout_codes & set(df_all['s_state'].unique())
print(f"  Codes present in data: {found_codes}")

# ── Geographic + temporal split ───────────────────────────────────────────────
is_holdout = df_all['s_state'].isin(holdout_codes)
mask_train = (~is_holdout) & (df_all['t_year'] <= 2021)
mask_test  = is_holdout    & (df_all['t_year'] == 2023)

df_geo_train = df_all[mask_train]
df_geo_test  = df_all[mask_test]
print(f"\n  Geo-train : {len(df_geo_train):,}  severe={df_geo_train['label'].mean()*100:.2f}%")
print(f"  Geo-test  : {len(df_geo_test):,}   severe={df_geo_test['label'].mean()*100:.2f}%")

# Use original 32 features (without target-encoded street/city) for geo model
# so results are comparable to primary model
X_geo_train = df_geo_train[FEAT_USA_V1].values.astype(np.float32)
y_geo_train = df_geo_train['label'].values
X_geo_test  = df_geo_test[FEAT_USA_V1].values.astype(np.float32)
y_geo_test  = df_geo_test['label'].values

no_skill_geo = float(y_geo_test.mean())
print(f"  No-skill AUPRC (geo): {no_skill_geo:.4f}")

# ── Train with same hyperparameters ───────────────────────────────────────────
neg_pos_ratio = float((y_geo_train == 0).sum()) / max(float((y_geo_train == 1).sum()), 1)
val_n  = min(80_000, int(len(X_geo_train) * 0.1))
X_gv   = X_geo_train[-val_n:]; y_gv = y_geo_train[-val_n:]
X_gt   = X_geo_train[:-val_n]; y_gt = y_geo_train[:-val_n]

dtrain_g = xgb.DMatrix(X_gt, label=y_gt)
dval_g   = xgb.DMatrix(X_gv, label=y_gv)
dtest_g  = xgb.DMatrix(X_geo_test)

params_geo = {
    **{k: v for k, v in xgb_best_params.items()
       if k not in ('n_estimators', 'early_stopping_rounds')},
    'scale_pos_weight': neg_pos_ratio,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist', 'device': 'cuda',
    'seed': SEED,
}
n_rounds = int(xgb_best_params.get('n_estimators', 1100))

print(f"\nTraining geo XGBoost ({n_rounds} rounds, early stop 50) ...")
t0 = time.time()
geo_model = xgb.train(
    params_geo, dtrain_g,
    num_boost_round=n_rounds,
    evals=[(dval_g, 'val')],
    early_stopping_rounds=50,
    verbose_eval=200,
)
print(f"  Done in {time.time()-t0:.0f}s")

# ── Evaluate ───────────────────────────────────────────────────────────────────
preds_geo = geo_model.predict(dtest_g)
auprc_geo = average_precision_score(y_geo_test, preds_geo)
auroc_geo = roc_auc_score(y_geo_test, preds_geo)
lift_geo  = auprc_geo / no_skill_geo

print("\n" + "=" * 60)
print("GEOGRAPHIC HOLD-OUT RESULTS")
print(f"  Hold-out states         : {', '.join(HOLDOUT_STATES)}")
print(f"  Geo-test records (2023) : {len(X_geo_test):,}")
print(f"  No-skill AUPRC          : {no_skill_geo:.4f}")
print(f"  Geo hold-out AUPRC      : {auprc_geo:.4f}  (lift = {lift_geo:.2f}×)")
print(f"  Geo hold-out AUROC      : {auroc_geo:.4f}")
print(f"  Primary model AUPRC     : 0.1306  (same-geography 2023 test)")
print(f"  Primary lift            : {0.1306/0.027:.1f}× over 0.027 baseline")
print()
if lift_geo >= 3.0:
    verdict = "STRONG — genuinely transferable spatial risk signal"
elif lift_geo >= 1.5:
    verdict = "PARTIAL — some transferable signal alongside memorisation"
elif lift_geo >= 1.2:
    verdict = "WEAK — predominantly location memorisation"
else:
    verdict = "NONE — confirms dominant location memorisation"
print(f"  Verdict: {verdict}")

geo_results = {
    'holdout_states': HOLDOUT_STATES,
    'n_geo_train': int(len(X_geo_train)),
    'n_geo_test': int(len(X_geo_test)),
    'no_skill_auprc': round(no_skill_geo, 4),
    'geo_holdout_auprc': round(auprc_geo, 4),
    'geo_holdout_auroc': round(auroc_geo, 4),
    'lift_over_noskill': round(lift_geo, 3),
    'primary_model_auprc': 0.1306,
    'primary_noskill': 0.027,
    'primary_lift': round(0.1306/0.027, 2),
    'verdict': verdict,
    'generated_at': str(datetime.now()),
}
with open(LOGS_DIR / 'geo_holdout_results.json', 'w') as f:
    json.dump(geo_results, f, indent=2)
print(f"\nSaved: {LOGS_DIR / 'geo_holdout_results.json'} ✓")
print("P10 complete ✓")


P10 — Geographic Hold-Out Experiment
Loading train + val + test parquets ...
  Combined: 6,985,228 rows
  Holdout codes: ['CA', 'TX', 'FL'] → {8, 42, 3}
  Codes present in data: {8, 42, 3}

  Geo-train : 3,659,419  severe=3.49%
  Geo-test  : 66,791   severe=0.57%
  No-skill AUPRC (geo): 0.0057

Training geo XGBoost (1317 rounds, early stop 50) ...


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [08:17:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [08:17:27] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[0]	val-aucpr:0.35395
[84]	val-aucpr:0.37358
  Done in 19s

GEOGRAPHIC HOLD-OUT RESULTS
  Hold-out states         : CA, TX, FL
  Geo-test records (2023) : 66,791
  No-skill AUPRC          : 0.0057
  Geo hold-out AUPRC      : 0.0164  (lift = 2.90×)
  Geo hold-out AUROC      : 0.6342
  Primary model AUPRC     : 0.1306  (same-geography 2023 test)
  Primary lift            : 4.8× over 0.027 baseline

  Verdict: PARTIAL — some transferable signal alongside memorisation

Saved: /content/drive/MyDrive/SAE-XCrash/logs/wp2_v2/geo_holdout_results.json ✓
P10 complete ✓


---
## Step 5 — Summary

In [ ]:
print("NB02_Modeling_Baselines_v2 COMPLETE")
print("=" * 60)
print(f"  Outputs: {LOGS_DIR}")
print()
print("  Files written:")
for f in sorted(LOGS_DIR.iterdir()):
    print(f"    {f.name}  ({f.stat().st_size/1e3:.1f} KB)")
print()
print("  Next: share geo_holdout_results.json + ablation_lgbm_results.json")
print("        to update v26 paper with results.")


NB02_Modeling_Baselines_v2 COMPLETE
  Outputs: /content/drive/MyDrive/SAE-XCrash/logs/wp2_v2

  Files written:
    ablation_lgbm_results.json  (0.6 KB)
    ablation_xgb_results.json  (0.6 KB)
    geo_holdout_results.json  (0.4 KB)

  Next: share geo_holdout_results.json + ablation_lgbm_results.json
        to update v26 paper with results.
